In [ ]:
# ============================================================
# 04 - FEATURE ENGINEERING
# Pune CPCB PM2.5 Forecasting - ALL STATIONS
# ============================================================

import os
import pandas as pd
import numpy as np


# ============================================================
# 1. FIND PROJECT DIRECTORY
# ============================================================

BASE_DIR = os.getcwd()

if os.path.basename(BASE_DIR).lower() == "notebooks":
    BASE_DIR = os.path.dirname(BASE_DIR)


# ============================================================
# 2. FILE PATHS
# ============================================================

INPUT_FILE = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "pune_cpcb_cleaned_hourly.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "pune_cpcb_features.csv"
)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("=" * 70)
print("PUNE CPCB FEATURE ENGINEERING")
print("=" * 70)

print("\nLoading:")
print(INPUT_FILE)

df = pd.read_csv(INPUT_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("\nOriginal shape:", df.shape)

print(
    "Stations:",
    df["station_name"].nunique()
)


# ============================================================
# 4. SORT DATA
# ============================================================

df = df.sort_values(
    ["station_name", "timestamp"]
).reset_index(drop=True)


# ============================================================
# 5. TIME FEATURES
# ============================================================

print("\nCreating time features...")

df["hour"] = df["timestamp"].dt.hour

df["day"] = df["timestamp"].dt.day

df["day_of_week"] = df["timestamp"].dt.dayofweek

df["month"] = df["timestamp"].dt.month

df["week_of_year"] = (
    df["timestamp"].dt.isocalendar().week.astype(int)
)

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)


# ============================================================
# 6. CYCLICAL TIME FEATURES
# ============================================================

# Hour
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# Day of week
df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

# Month
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)


# ============================================================
# 7. PM2.5 LAG FEATURES
# ============================================================

print("Creating PM2.5 lag features...")

lag_values = [
    1,
    2,
    3,
    6,
    12,
    24,
    48,
    72
]

for lag in lag_values:

    df[f"pm25_lag_{lag}h"] = (
        df.groupby("station_name")["pm25"]
        .shift(lag)
    )


# ============================================================
# 8. POLLUTANT LAG FEATURES
# ============================================================

print("Creating pollutant lag features...")

pollutant_columns = [
    "pm10",
    "no2",
    "so2",
    "co",
    "ozone"
]

pollutant_columns = [
    col for col in pollutant_columns
    if col in df.columns
]

for column in pollutant_columns:

    df[f"{column}_lag_1h"] = (
        df.groupby("station_name")[column]
        .shift(1)
    )


# ============================================================
# 9. PM2.5 ROLLING FEATURES
# ============================================================

print("Creating rolling features...")

# IMPORTANT:
# shift(1) means the current PM2.5 value is NOT included.
# This prevents data leakage.

grouped_pm25 = (
    df.groupby("station_name")["pm25"]
    .shift(1)
)

for window in [3, 6, 12, 24]:

    df[f"pm25_rolling_mean_{window}h"] = (
        grouped_pm25
        .groupby(df["station_name"])
        .transform(
            lambda x: x.rolling(
                window
            ).mean()
        )
    )


# ============================================================
# 10. PM2.5 ROLLING MAX
# ============================================================

for window in [6, 24]:

    df[f"pm25_rolling_max_{window}h"] = (
        grouped_pm25
        .groupby(df["station_name"])
        .transform(
            lambda x: x.rolling(
                window
            ).max()
        )
    )


# ============================================================
# 11. PM2.5 CHANGE FEATURES
# ============================================================

print("Creating PM2.5 change features...")

df["pm25_change_1h"] = (
    df.groupby("station_name")["pm25"]
    .diff(1)
)

df["pm25_change_3h"] = (
    df.groupby("station_name")["pm25"]
    .diff(3)
)

df["pm25_change_24h"] = (
    df.groupby("station_name")["pm25"]
    .diff(24)
)


# ============================================================
# 12. WEATHER FEATURES
# ============================================================

weather_columns = [
    "temperature",
    "humidity",
    "wind_speed",
    "wind_direction",
    "rainfall"
]

weather_columns = [
    col for col in weather_columns
    if col in df.columns
]

print("\nWeather features used:")
print(weather_columns)


# ============================================================
# 13. WIND DIRECTION CYCLICAL FEATURES
# ============================================================

if "wind_direction" in df.columns:

    df["wind_direction_sin"] = np.sin(
        np.deg2rad(df["wind_direction"])
    )

    df["wind_direction_cos"] = np.cos(
        np.deg2rad(df["wind_direction"])
    )


# ============================================================
# 14. POLLUTANT RATIOS
# ============================================================

print("\nCreating pollutant ratio features...")

if "pm10" in df.columns:

    df["pm25_pm10_ratio"] = (
        df["pm25"] /
        df["pm10"].replace(0, np.nan)
    )

if "no2" in df.columns:

    df["pm25_no2_ratio"] = (
        df["pm25"] /
        df["no2"].replace(0, np.nan)
    )


# ============================================================
# 15. REMOVE INFINITE VALUES
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 16. REMOVE ROWS WITH MISSING TARGET
# ============================================================

before = len(df)

df = df.dropna(
    subset=["pm25"]
).reset_index(drop=True)

after = len(df)

print(
    "\nRows removed because PM2.5 was missing:",
    before - after
)


# ============================================================
# 17. REMOVE INITIAL LAG/ROLLING ROWS
# ============================================================

# The first 72 hours of each station don't have all lag values.
# Remove rows where the 72-hour lag is unavailable.

before = len(df)

df = df.dropna(
    subset=["pm25_lag_72h"]
).reset_index(drop=True)

after = len(df)

print(
    "Rows removed because 72-hour history was unavailable:",
    before - after
)


# ============================================================
# 18. CHECK FEATURE DATASET
# ============================================================

print("\n" + "=" * 70)
print("FEATURE DATASET")
print("=" * 70)

print("\nShape:", df.shape)

print(
    "Stations:",
    df["station_name"].nunique()
)

print(
    "\nNumber of features:",
    len(df.columns)
)


# ============================================================
# 19. DISPLAY FEATURES
# ============================================================

print("\nFeatures created:")

for number, column in enumerate(
    df.columns,
    start=1
):

    print(
        number,
        ".",
        column
    )


# ============================================================
# 20. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES AFTER FEATURE ENGINEERING")
print("=" * 70)

missing = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(missing[missing > 0])


# ============================================================
# 21. CHECK SAMPLE
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE FEATURE DATA")
print("=" * 70)

display(df.head())


# ============================================================
# 22. SAVE FEATURE DATASET
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("✅ FEATURE ENGINEERING COMPLETED")
print("=" * 70)

print("\nSaved file:")
print(OUTPUT_FILE)

print("\nFinal dataset shape:")
print(df.shape)